In [1]:
import jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", False)
import os, json, numpy as np, matplotlib.pyplot as plt
import sys

sys.path.insert(0, '../src')
sys.path.insert(0, '../scripts/experiments_large_scale/')

import run_LowRank

from run_LowRank import (seed_everything, get_device,
            stratified_equal_halves
)

In [2]:
"""
Timepoints, indexed by AnnData index
"""
timepoints_across_datasets =  [['E9.25', 'E9.5', 'E9.75', 'E10.0']]

'''
timepoints_across_datasets =  [['E8.5', 'E8.75', 'E9.0', 'E9.25', 'E9.5', 'E9.75', 'E10.0', 'E10.25', 'E10.5', 'E10.75'], \
                               ['E11.0', 'E11.25', 'E11.5', 'E11.75', 'E12.0', 'E12.25', 'E12.5', 'E12.75', 'E13.0', 'E13.25', 'E13.5', 'E13.75'], \
                               ['E14.0', 'E14.25', 'E14.333', 'E14.75', 'E15.0', 'E15.25', 'E15.5', 'E15.75', 'E16.0', 'E16.25', 'E16.5', 'E16.75'], \
                               ['E17.0', 'E17.25', 'E17.5', 'E17.75', 'E18.0', 'E18.25', 'E18.5', 'E18.75']]
'''

"""
For simplicity, we choose the first replicate for each timepoint as our dataset.
"""
replicates_across_datasets = [['embryo_20', 'embryo_24', 'embryo_28', 'embryo_30']]

'''
replicates_across_datasets = [['embryo_11', 'embryo_14', 'embryo_16', 'embryo_20', 'embryo_24', 'embryo_28', 'embryo_30', 'embryo_33', 'embryo_34', 'embryo_35'], \
                              ['embryo_36', 'embryo_37', 'embryo_38', 'embryo_39', 'embryo_40', 'embryo_41', 'embryo_42', 'embryo_43', 'embryo_45', 'embryo_47', 'embryo_49', 'embryo_52'], \
                              ['embryo_53', 'embryo_54', 'embryo_55', 'embryo_56', 'embryo_57', 'embryo_58', 'embryo_59', 'embryo_60', 'embryo_61', 'embryo_62', 'embryo_63', 'embryo_64'], \
                              ['embryo_65', 'embryo_66', 'embryo_67', 'embryo_68', 'embryo_69', 'embryo_70', 'embryo_71', 'embryo_72']]
'''


"\nreplicates_across_datasets = [['embryo_11', 'embryo_14', 'embryo_16', 'embryo_20', 'embryo_24', 'embryo_28', 'embryo_30', 'embryo_33', 'embryo_34', 'embryo_35'],                               ['embryo_36', 'embryo_37', 'embryo_38', 'embryo_39', 'embryo_40', 'embryo_41', 'embryo_42', 'embryo_43', 'embryo_45', 'embryo_47', 'embryo_49', 'embryo_52'],                               ['embryo_53', 'embryo_54', 'embryo_55', 'embryo_56', 'embryo_57', 'embryo_58', 'embryo_59', 'embryo_60', 'embryo_61', 'embryo_62', 'embryo_63', 'embryo_64'],                               ['embryo_65', 'embryo_66', 'embryo_67', 'embryo_68', 'embryo_69', 'embryo_70', 'embryo_71', 'embryo_72']]\n"

In [3]:
import os, json, sys, gc
import numpy as np
import pandas as pd
import scanpy as sc
import single_cell
import importlib
import monge_rotate_lr

importlib.reload(single_cell)
importlib.reload(run_LowRank)
importlib.reload(monge_rotate_lr)

# Cell-type labels (index must contain 'cell_id' to join on)
df_cell = pd.read_csv("/scratch/gpfs/ph3641/hm_ot/df_cell.csv").set_index("cell_id")
# Whether minor subsampling is needed -- for HiRef to work need many divisors, so decrease n slightly to get this
subsample_to_nonprime_tp = [ True, True, True, True]
max_Qs = [ 100, 100, 100, 100]
max_ranks=[ 5000, 5000, 5000, 5000]

all_results = []

for i in range(len(timepoints_across_datasets)):
    # --- load dataset i ---
    filehandle_ME = f"/scratch/gpfs/ph3641/hm_ot/adata_JAX_dataset_{i+1}.h5ad"
    adata = sc.read_h5ad(filehandle_ME, backed="r")

    timepoints = timepoints_across_datasets[i]
    replicates = replicates_across_datasets[i]

    for tp in range(len(timepoints)):
        subset_adata = None
        
        if (tp + 1) < len(timepoints):
            # pair within the same dataset
            t1, t2 = timepoints[tp], timepoints[tp + 1]
            r1, r2 = replicates[tp], replicates[tp + 1]
            print(f"[intra] Aligning timepoint {t1} ({r1}) → {t2} ({r2})")

            # subset by original string fields; move to memory
            subset_adata = adata[
                (adata.obs["day"].isin([t1, t2])) &
                (adata.obs["embryo_id"].isin([r1, r2]))
            ].to_memory()

        elif (i + 1) < len(timepoints_across_datasets):
            # cross to the next dataset's first timepoint
            timepoints2 = timepoints_across_datasets[i + 1]
            replicates2 = replicates_across_datasets[i + 1]
            t1, t2 = timepoints[tp], timepoints2[0]
            r1, r2 = replicates[tp], replicates2[0]
            print(f"[inter] Aligning timepoint {t1} ({r1}) → {t2} ({r2})")

            # last slice from current dataset to memory
            adata1 = adata[
                (adata.obs["day"] == t1) &
                (adata.obs["embryo_id"] == r1)
            ].to_memory()

            # open next dataset and slice first pair to memory
            filehandle_ME2 = f"/scratch/gpfs/ph3641/hm_ot/adata_JAX_dataset_{i+2}.h5ad"
            adata2_full = sc.read_h5ad(filehandle_ME2, backed="r")
            adata2 = adata2_full[
                (adata2_full.obs["day"] == t2) &
                (adata2_full.obs["embryo_id"] == r2)
            ].to_memory()
            del adata2_full; gc.collect()

            # concatenate the two to one in-memory object
            subset_adata = adata1.concatenate(adata2, index_unique=None)
            del adata1, adata2; gc.collect()
        else:
            # nothing to align at the tail
            continue

        # skip if empty (can happen with missing replicates)
        if subset_adata.n_obs == 0:
            print("  -> empty subset, skipping")
            continue

        # integrate labels (expects 'cell_id' present as a column)
        if "cell_id" not in subset_adata.obs.columns:
            print("  -> missing 'cell_id' in obs; cannot join labels, skipping")
            del subset_adata; gc.collect()
            continue

        subset_adata.obs = subset_adata.obs.set_index("cell_id", drop=False)
        subset_adata.obs = subset_adata.obs.join(df_cell[["celltype_update"]], how="left")

        # convert 'day' to ordered numeric category (for plotting/metadata), AFTER subsetting
        # keep original strings for selection logic above
        try:
            day_num = subset_adata.obs["day"].astype(str).str.extract(r"(\d+(?:\.\d+)?)").astype(float)[0]
            subset_adata.obs["day_num"] = day_num
            subset_adata.obs["day_num"] = pd.Categorical(subset_adata.obs["day_num"], ordered=True)
        except Exception as e:
            print(f"  -> warning: could not parse day to numeric: {e}")

        # --- run LR-OT methods + evaluation ---
        try:
            df_out = single_cell.run_pairwise_eval(
                subset_adata, t1, t2,
                methods=( "monge_conj", "frlc", "lot" ),
                label_col="celltype_update",
                seed=0, use_pca=True, pca_key="X_pca", n_comps=50,
                max_rank=None,
                subsample_to_nonprime=subsample_to_nonprime_tp[tp],
                hiref_max_Q=max_Qs[tp], hiref_max_rank=max_ranks[tp],
                hiref_iters=300
                # or set to a fixed rank, e.g., 10
            )
            print(df_out)
            all_results.append(df_out)
        except Exception as e:
            print(f"  -> evaluation failed: {e}")

        # free memory aggressively
        del subset_adata; gc.collect()
    
    # close backed AnnData view (let file handle go)
    del adata; gc.collect()

# save combined results
if all_results:
    all_results = pd.concat(all_results, ignore_index=True)
    save_csv = "sc_timepair_lr_ot_results_2.csv"
    all_results.to_csv(save_csv, index=False)
    print("Saved:", save_csv)
else:
    print("No results produced.")


[intra] Aligning timepoint E9.25 (embryo_20) → E9.5 (embryo_24)
Subsampled to n=75600 from n=83059 [needed for Hierarchical Refinement]
Optimized rank-annealing schedule: [840, 90]


2025-09-21 23:37:53.943 | INFO     | monge_rotate_lr:monge_rotation_kmeans_LR:184 - Initialization Costs: (0.17750776977221022, 0.17671990964548212)


Iteration: 0


2025-09-21 23:38:37.889923: W external/xla/xla/hlo/transforms/simplifiers/hlo_rematerialization.cc:3023] Can't reduce memory use below 27.94GiB (30001577444 bytes) by rematerialization; only reduced to 85.24GiB (91531001738 bytes), down from 85.25GiB (91532211338 bytes) originally
2025-09-21 23:38:50.106611: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 42.59GiB (rounded to 45727087616)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-09-21 23:38:50.106771: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] *___________________________________________________________________________________________________
E0921 23:38:50.106785 2182318 pjrt_stream_executor_client.cc:2916] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory whil

  timepoint_A timepoint_B      method  rank   ot_cost     A_AMI     A_ARI  \
0       E9.25        E9.5  monge_conj    67  0.411002  0.561973  0.190665   
1       E9.25        E9.5        frlc    67  0.431250  0.484287  0.128585   
2       E9.25        E9.5         lot    67       NaN       NaN       NaN   

      B_AMI     B_ARI       CMA  runtime_sec  n_cells  \
0  0.566709  0.193688  0.565059   470.615793  75600.0   
1  0.488075  0.130265  0.440566    33.918150  75600.0   
2       NaN       NaN       NaN          NaN      NaN   

                                               error  
0                                                NaN  
1                                                NaN  
2  RESOURCE_EXHAUSTED: Out of memory while trying...  
[intra] Aligning timepoint E9.5 (embryo_24) → E9.75 (embryo_28)
Subsampled to n=131040 from n=137852 [needed for Hierarchical Refinement]
Optimized rank-annealing schedule: [1365, 96]


...

Subsampled to n=120960 from n=126872 [needed for Hierarchical Refinement]
Optimized rank-annealing schedule: [1260, 96]


2025-09-22 00:12:17.970 | INFO     | monge_rotate_lr:monge_rotation_kmeans_LR:184 - Initialization Costs: (0.17083341504722205, 0.17007023616203243)


Iteration: 0


2025-09-22 00:13:22.020941: W external/xla/xla/hlo/transforms/simplifiers/hlo_rematerialization.cc:3023] Can't reduce memory use below 27.79GiB (29837253132 bytes) by rematerialization; only reduced to 218.17GiB (234254366938 bytes), down from 218.17GiB (234256302298 bytes) originally
2025-09-22 00:13:34.363951: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 109.02GiB (rounded to 117057730560)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-09-22 00:13:34.364156: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] ***_________________________________________________________________________________________________
E0922 00:13:34.364169 2182318 pjrt_stream_executor_client.cc:2916] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memor

  timepoint_A timepoint_B      method  rank   ot_cost     A_AMI     A_ARI  \
0       E9.75       E10.0  monge_conj    77  0.360656  0.558994  0.180390   
1       E9.75       E10.0        frlc    77  0.379064  0.501808  0.129791   
2       E9.75       E10.0         lot    77       NaN       NaN       NaN   

      B_AMI     B_ARI       CMA  runtime_sec   n_cells  \
0  0.560136  0.180638  0.474530   741.912632  120960.0   
1  0.502072  0.130424  0.436657    52.020052  120960.0   
2       NaN       NaN       NaN          NaN       NaN   

                                               error  
0                                                NaN  
1                                                NaN  
2  RESOURCE_EXHAUSTED: Out of memory while trying...  
Saved: sc_timepair_lr_ot_results_2.csv
